# Part 1: Full-Text Container Shipping Disruption Scoring with Gemma 2 9B IT


## Workflow

1. Configure the CSV path, source columns, Gemma 2 model path, and `CHUNK_NUMBER`.
2. Load and validate the complete CSV.
3. Create a chunk manifest and optional 700-row raw chunk files.
4. Load Gemma 2 9B IT in 4-bit mode.
5. Run a smoke test.
6. Score the selected 700-row chunk with resumable checkpointing.
7. Inspect distributions and failures.
8. Package the chunk outputs.
9. After all run chunks are complete, merge them with the final consolidation section.


## 1. Environment setup

Enable a Kaggle GPU before running the model-loading cell. The default configuration uses 4-bit quantization to fit reliably on a Kaggle T4 GPU.

In [1]:
from __future__ import annotations
!pip install -q -U "transformers>=4.51.0,<5" accelerate bitsandbytes ftfy pandas

import gc
import hashlib
import json
import logging
import math
import os
import random
import re
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
from ftfy import fix_text
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoModelForCausalLM,
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 589.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.5 which is

## 2. Configuration

Normally, only this cell needs editing for each Kaggle run.

`CHUNK_NUMBER` is **1-based**. With approximately 6,500 rows and `RUN_CHUNK_SIZE = 700`, the dataset will require about 10 run chunks.

In [2]:
# ------------------------------------------------------------------
# Input CSV and source columns
# ------------------------------------------------------------------
INPUT_CSV = Path("/kaggle/input/datasets/mdmuqtadirfuad/llm-maritime-disruption/pre_llm_kept.csv")

# These names match the supplied dataset sample.
ID_COLUMN: str | None = "article_id"
TEXT_COLUMN: str | None = "retrieved_text"
HEADLINE_COLUMN: str | None = "headline"
DATE_COLUMN: str | None = "retrieved_publish_date"

# Standard CSV is comma-separated. Change only if your actual file differs.
CSV_SEPARATOR = ","
CSV_ENCODING_CANDIDATES = ("utf-8", "utf-8-sig", "cp1252", "latin-1")

# Repair common encoding artefacts 
# The original source columns remain unchanged.
REPAIR_TEXT_ENCODING = True

# ------------------------------------------------------------------
# Dataset-level Kaggle chunking
# ------------------------------------------------------------------
RUN_CHUNK_SIZE = 700
CHUNK_NUMBER = 10             
WRITE_ALL_RAW_CHUNKS = True

# ------------------------------------------------------------------
# Model
# ------------------------------------------------------------------
# Gemma 2 9B IT is used because its capacity is much closer to the
MODEL_NAME = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2"
MODEL_SOURCE = MODEL_NAME

USE_4BIT = True
RANDOM_SEED = 42

# These values are intentionally identical to the Qwen2.5/Qwen3 Q1 runs.
MAX_TEXT_TOKENS_PER_WINDOW = 6000
WINDOW_TOKEN_OVERLAP = 200
MAX_NEW_TOKENS = 1200

# ------------------------------------------------------------------
# Reliability and persistence
# ------------------------------------------------------------------
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 3
CHECKPOINT_EVERY_ARTICLES = 10
SAVE_RAW_MODEL_RESPONSE = True

WORKING_DIR = Path("/kaggle/working/shipping_disruption_scoring_gemma2_9b")
RAW_CHUNKS_DIR = WORKING_DIR / "raw_chunks"
RUN_DIR = WORKING_DIR / f"chunk_{CHUNK_NUMBER:03d}"

CHECKPOINT_FILE = RUN_DIR / f"chunk_{CHUNK_NUMBER:03d}_checkpoint.csv"
FINAL_OUTPUT_FILE = RUN_DIR / f"scored_chunk_{CHUNK_NUMBER:03d}.csv"
AUDIT_JSONL_FILE = RUN_DIR / f"chunk_{CHUNK_NUMBER:03d}_audit.jsonl"
RUN_LOG_FILE = RUN_DIR / f"chunk_{CHUNK_NUMBER:03d}_run.log"
FAILURES_FILE = RUN_DIR / f"chunk_{CHUNK_NUMBER:03d}_failures.csv"
ARTIFACT_ZIP_BASE = WORKING_DIR / f"chunk_{CHUNK_NUMBER:03d}_artifacts"

RUN_DIR.mkdir(parents=True, exist_ok=True)
RAW_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Reproducibility and logging

The run log records configuration, dataset validation, progress, retries, checkpoint writes, failures, and completion summaries. The JSONL audit file adds one machine-readable record per article or article window.

In [3]:
def set_reproducibility(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_logger(log_file: Path) -> logging.Logger:
    logger = logging.getLogger("shipping_disruption_scoring")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    if logger.handlers:
        logger.handlers.clear()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%dT%H:%M:%S",
    )

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def append_jsonl(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(payload, ensure_ascii=False, default=str) + "\n")


set_reproducibility()
logger = configure_logger(RUN_LOG_FILE)

logger.info("Run initialized")
logger.info("Input CSV: %s", INPUT_CSV)
logger.info("Run chunk size: %s", RUN_CHUNK_SIZE)
logger.info("Selected chunk number: %s", CHUNK_NUMBER)
logger.info("Model: %s", MODEL_NAME)
logger.info("Model source: %s", MODEL_SOURCE)
logger.info("Use 4-bit: %s", USE_4BIT)
logger.info("CUDA available: %s", torch.cuda.is_available())
logger.info("GPU count: %s", torch.cuda.device_count())

for gpu_id in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(gpu_id)
    logger.info(
        "GPU %s: %s, %.2f GB",
        gpu_id,
        torch.cuda.get_device_name(gpu_id),
        properties.total_memory / 1e9,
    )

2026-07-28T05:59:53 | INFO | Run initialized
2026-07-28T05:59:53 | INFO | Input CSV: /kaggle/input/datasets/mdmuqtadirfuad/llm-maritime-disruption/pre_llm_kept.csv
2026-07-28T05:59:53 | INFO | Run chunk size: 700
2026-07-28T05:59:53 | INFO | Selected chunk number: 10
2026-07-28T05:59:53 | INFO | Model: /kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2
2026-07-28T05:59:53 | INFO | Model source: /kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2
2026-07-28T05:59:53 | INFO | Use 4-bit: True
2026-07-28T05:59:53 | INFO | CUDA available: True
2026-07-28T05:59:53 | INFO | GPU count: 2
2026-07-28T05:59:53 | INFO | GPU 0: Tesla T4, 15.64 GB
2026-07-28T05:59:53 | INFO | GPU 1: Tesla T4, 15.64 GB


## 4. Taxonomy and severity rubric

Each theme is scored independently. A high score in one theme does not automatically force a high score in another theme.

### Common 0–4 severity scale

| Score | Interpretation |
|---:|---|
| 0 | No evidence of this disruption theme |
| 1 | Weak, indirect, anticipated, or minor evidence; little demonstrated impact |
| 2 | Clear but moderate or localized disruption with limited scope, duration, or consequences |
| 3 | Major disruption with substantial operational or economic consequences |
| 4 | Severe or systemic disruption with shutdown, blockage, exceptional scarcity/cost, broad international spread, or prolonged consequences |

The prompt requires evidence from the article and prohibits scoring ordinary shipping activity as disruption.

In [4]:
THEME_COLUMNS = [
    "port_disruption_score",
    "global_container_movement_disruption_score",
    "freight_cost_disruption_score",
    "supply_disruption_score",
    "production_disruption_score",
    "covid19_related_disruption_score",
    "demand_disruption_score",
]

THEME_KEYS = {
    "port_disruption": "port_disruption_score",
    "global_container_movement_disruption": "global_container_movement_disruption_score",
    "freight_cost_disruption": "freight_cost_disruption_score",
    "supply_disruption": "supply_disruption_score",
    "production_disruption": "production_disruption_score",
    "covid19_related_disruption": "covid19_related_disruption_score",
    "demand_disruption": "demand_disruption_score",
}

TAXONOMY_TEXT = '''
CONTAINER SHIPPING DISRUPTIONS DUE TO COVID-19 — CLASSIFICATION TAXONOMY

AGGREGATE DIMENSION 1: SHIPPING DISRUPTIONS

1. Port Disruption
   First-order codes: trapped containers, port congestion, port shutdowns.
   Include material terminal congestion, berth delays, port closures, inability
   to load/unload, cargo trapped at ports, and severe port-capacity constraints.

2. Global Container Movement Disruptions
   First-order codes: container supply imbalance, lag from delayed shipping.
   Include container shortages or surpluses, equipment imbalance, blank sailings,
   stranded cargo, network delays, schedule unreliability, rerouting, and severe
   interruption of container circulation.

3. Freight Cost Disruptions
   First-order codes: increased freight price, rising inflation.
   Include unusual or disruption-driven freight-rate increases, surcharges,
   transport-cost escalation, and inflationary effects explicitly linked to
   container shipping or freight disruption.

AGGREGATE DIMENSION 2: ECONOMIC DISRUPTIONS

4. Supply Disruptions
   First-order codes: supply choke-points, supply service storage, shipping alliances.
   Include bottlenecks in the supply network, constrained logistics services,
   disruption-related storage problems, and alliance/service-network actions that
   materially interrupt supply-chain flow.

5. Production Disruptions
   First-order codes: factory shutdowns, shortage of labour.
   Include plant closures, curtailed output, inability to obtain production inputs,
   and labour shortages that materially affect production or shipping-linked supply.

6. COVID-19 Related Disruptions
   First-order codes: lockdowns, travel restrictions, transportation restrictions.
   Score only when COVID-19, coronavirus, pandemic restrictions, quarantine, health
   rules, or their direct consequences cause or materially intensify disruption.
   A passing reference to COVID-19 is not sufficient.

7. Demand Disruptions
   First-order codes: changes in consumer demands, changes in demand of shipping
   volume, choke-points in Asia.
   Include sharp demand collapse or surge, material changes in container volumes,
   pandemic-driven consumption shifts, and demand-related Asian bottlenecks.
'''.strip()

SEVERITY_RUBRIC = '''
Apply this scale independently to every theme:
0 = absent: no article evidence for this theme.
1 = weak/minor: indirect, anticipated, small, or incidental disruption; impact is
    limited or not demonstrated.
2 = moderate: clear realized disruption, but localized, temporary, or limited in
    operational/economic scope.
3 = major: substantial realized disruption, large operational consequences,
    sustained effects, or impact across several firms, ports, routes, or markets.
4 = severe/systemic: shutdown, blockage, extreme congestion or imbalance,
    exceptional freight-cost escalation, widespread international impact,
    prolonged crisis, or a central disruption event.

Important controls:
- Score the event described, not the number of times a keyword appears.
- Do not treat normal investment, expansion, safety discussion, market commentary,
  or routine shipping operations as a disruption unless material disruption is shown.
- Do not infer facts absent from the article.
- Scores must be integers from 0 through 4.
- Use the complete supplied article segment as evidence.
'''.strip()

SYSTEM_PROMPT = f'''
You are a rigorous maritime supply-chain research coder.

Your task is to score a news article under seven disruption themes. Use only the
article evidence and the taxonomy below. The seven scores are independent.

{TAXONOMY_TEXT}

{SEVERITY_RUBRIC}

Return exactly one valid JSON object and no markdown. Use this exact schema:

{{
  "scores": {{
    "port_disruption": 0,
    "global_container_movement_disruption": 0,
    "freight_cost_disruption": 0,
    "supply_disruption": 0,
    "production_disruption": 0,
    "covid19_related_disruption": 0,
    "demand_disruption": 0
  }},
  "evidence": {{
    "port_disruption": "",
    "global_container_movement_disruption": "",
    "freight_cost_disruption": "",
    "supply_disruption": "",
    "production_disruption": "",
    "covid19_related_disruption": "",
    "demand_disruption": ""
  }},
  "overall_reasoning": ""
}}

Evidence rules:
- For every score above 0, provide a short paraphrase of the supporting evidence.
- For a score of 0, use an empty string.
- Do not copy long passages.
- Keep overall_reasoning to no more than two concise sentences.
- Never omit the scores, evidence, or overall_reasoning top-level keys.
- If all seven scores are 0, set overall_reasoning to:
  "No taxonomy-defined disruption was identified in this article window."
'''.strip()

## 5. Response parsing and validation

The seven score keys, integer 0–4 limits, and supporting evidence for every
positive score remain strict. Invalid scoring output is retried.

`overall_reasoning` is supplementary. When that field alone is missing, the
notebook now creates a deterministic explanation from the validated scores and
evidence, logs the fallback, and accepts the result without another model call.


In [5]:
def extract_json_object(text: str) -> dict[str, Any]:
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("No JSON object found in model response")

    parsed = json.loads(cleaned[start : end + 1])
    if not isinstance(parsed, dict):
        raise ValueError("Parsed JSON is not an object")
    return parsed


THEME_LABELS = {
    "port_disruption": "port disruption",
    "global_container_movement_disruption": "global container movement disruption",
    "freight_cost_disruption": "freight cost disruption",
    "supply_disruption": "supply disruption",
    "production_disruption": "production disruption",
    "covid19_related_disruption": "COVID-19-related disruption",
    "demand_disruption": "demand disruption",
}


def build_fallback_reasoning(
    normalized_scores: dict[str, int],
    normalized_evidence: dict[str, str],
) -> str:
    """Create an explanation only when the model omits overall_reasoning."""
    active_parts: list[str] = []

    for model_key, score_column in THEME_KEYS.items():
        score = int(normalized_scores[score_column])
        if score <= 0:
            continue

        evidence = normalized_evidence.get(f"{model_key}_evidence", "").strip()
        active_parts.append(
            f"{THEME_LABELS[model_key]}={score}: {evidence}"
        )

    if not active_parts:
        return "No taxonomy-defined disruption was identified in this article window."

    reasoning = "Validated theme evidence indicates " + "; ".join(active_parts) + "."
    return reasoning if len(reasoning) <= 1500 else reasoning[:1497] + "..."


def validate_model_result(payload: dict[str, Any]) -> dict[str, Any]:
    raw_scores = payload.get("scores")
    raw_evidence = payload.get("evidence")

    if not isinstance(raw_scores, dict):
        raise ValueError("Missing or invalid 'scores' object")
    if not isinstance(raw_evidence, dict):
        raise ValueError("Missing or invalid 'evidence' object")

    normalized_scores: dict[str, int] = {}
    normalized_evidence: dict[str, str] = {}

    for model_key, output_column in THEME_KEYS.items():
        if model_key not in raw_scores:
            raise ValueError(f"Missing score key: {model_key}")

        raw_value = raw_scores[model_key]
        if isinstance(raw_value, bool):
            raise ValueError(f"Boolean score is invalid for {model_key}")

        try:
            numeric_value = int(raw_value)
        except (TypeError, ValueError) as error:
            raise ValueError(f"Non-integer score for {model_key}: {raw_value}") from error

        if numeric_value < 0 or numeric_value > 4:
            raise ValueError(f"Score outside 0-4 for {model_key}: {numeric_value}")

        evidence_text = str(raw_evidence.get(model_key, "") or "").strip()
        if numeric_value > 0 and not evidence_text:
            raise ValueError(f"Positive score lacks evidence for {model_key}")

        normalized_scores[output_column] = numeric_value
        normalized_evidence[f"{model_key}_evidence"] = evidence_text

    overall_reasoning = str(payload.get("overall_reasoning", "") or "").strip()
    overall_reasoning_generated = not bool(overall_reasoning)

    if overall_reasoning_generated:
        overall_reasoning = build_fallback_reasoning(
            normalized_scores,
            normalized_evidence,
        )

    return {
        **normalized_scores,
        **normalized_evidence,
        "overall_reasoning": overall_reasoning,
        "overall_reasoning_generated": overall_reasoning_generated,
    }


def derived_summary(scores: dict[str, int]) -> dict[str, float | int]:
    shipping_values = [
        scores["port_disruption_score"],
        scores["global_container_movement_disruption_score"],
        scores["freight_cost_disruption_score"],
    ]
    economic_values = [
        scores["supply_disruption_score"],
        scores["production_disruption_score"],
        scores["covid19_related_disruption_score"],
        scores["demand_disruption_score"],
    ]
    all_values = shipping_values + economic_values

    return {
        "shipping_dimension_max": int(max(shipping_values)),
        "shipping_dimension_mean": float(np.mean(shipping_values)),
        "shipping_dimension_sum": int(sum(shipping_values)),
        "economic_dimension_max": int(max(economic_values)),
        "economic_dimension_mean": float(np.mean(economic_values)),
        "economic_dimension_sum": int(sum(economic_values)),
        "overall_disruption_max": int(max(all_values)),
        "overall_disruption_mean": float(np.mean(all_values)),
        "overall_disruption_sum": int(sum(all_values)),
        "active_theme_count": int(sum(value > 0 for value in all_values)),
    }

## 6. CSV loading, column validation, and chunk creation

The loader preserves every original source column. It creates internal fields only for stable processing and the model input.

In [6]:
def read_csv_robust(path: Path) -> tuple[pd.DataFrame, str]:
    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {path}")

    last_error: Exception | None = None
    for encoding in CSV_ENCODING_CANDIDATES:
        try:
            frame = pd.read_csv(
                path,
                sep=CSV_SEPARATOR,
                encoding=encoding,
                low_memory=False,
            )
            return frame, encoding
        except UnicodeDecodeError as error:
            last_error = error

    raise UnicodeError(
        f"Could not decode CSV using {CSV_ENCODING_CANDIDATES}"
    ) from last_error


def resolve_column(
    frame: pd.DataFrame,
    configured: str | None,
    candidates: Iterable[str],
    required: bool,
) -> str | None:
    if configured is not None:
        if configured not in frame.columns:
            raise KeyError(f"Configured column not found: {configured}")
        return configured

    lookup = {str(column).strip().lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    if required:
        raise KeyError(f"Could not resolve required column from: {list(candidates)}")
    return None


def normalize_model_text(value: Any) -> tuple[str, bool]:
    if pd.isna(value):
        return "", False

    original = str(value).strip()
    if not REPAIR_TEXT_ENCODING:
        return original, False

    repaired = fix_text(original)
    return repaired, repaired != original


def load_and_prepare_dataset(path: Path) -> pd.DataFrame:
    frame, encoding = read_csv_robust(path)
    logger.info("CSV loaded with encoding: %s", encoding)
    logger.info("Source rows: %s", len(frame))
    logger.info("Source columns: %s", list(frame.columns))

    id_column = resolve_column(
        frame,
        ID_COLUMN,
        ("article_id", "id", "record_id"),
        required=False,
    )
    text_column = resolve_column(
        frame,
        TEXT_COLUMN,
        ("retrieved_text", "full_text", "article_text", "text", "content"),
        required=True,
    )
    headline_column = resolve_column(
        frame,
        HEADLINE_COLUMN,
        ("headline", "retrieved_title", "title"),
        required=False,
    )
    date_column = resolve_column(
        frame,
        DATE_COLUMN,
        ("retrieved_publish_date", "date", "published_date", "publication_date"),
        required=False,
    )

    logger.info("Resolved ID column: %s", id_column)
    logger.info("Resolved text column: %s", text_column)
    logger.info("Resolved headline column: %s", headline_column)
    logger.info("Resolved date column: %s", date_column)

    prepared = frame.copy()
    prepared.insert(0, "_source_row_number", np.arange(len(prepared), dtype=int))

    if id_column is None:
        prepared["_article_id"] = prepared["_source_row_number"].map(
            lambda value: f"row_{value:07d}"
        )
    else:
        prepared["_article_id"] = prepared[id_column].astype("string").fillna("").str.strip()
        missing_id = prepared["_article_id"].eq("")
        prepared.loc[missing_id, "_article_id"] = prepared.loc[
            missing_id, "_source_row_number"
        ].map(lambda value: f"row_{value:07d}")

    prepared["_row_uid"] = (
        prepared["_article_id"].astype(str)
        + "__"
        + prepared["_source_row_number"].astype(str)
    )

    normalized = prepared[text_column].apply(normalize_model_text)
    prepared["_model_text"] = normalized.map(lambda pair: pair[0])
    prepared["_text_encoding_repaired"] = normalized.map(lambda pair: pair[1])

    if headline_column is None:
        prepared["_model_headline"] = ""
    else:
        prepared["_model_headline"] = (
            prepared[headline_column].astype("string").fillna("").str.strip()
        )

    if date_column is None:
        prepared["_model_date"] = ""
    else:
        prepared["_model_date"] = (
            prepared[date_column].astype("string").fillna("").str.strip()
        )

    prepared["_text_char_count"] = prepared["_model_text"].str.len()
    prepared["_text_word_count"] = prepared["_model_text"].str.split().str.len().fillna(0).astype(int)
    prepared["_has_usable_text"] = prepared["_model_text"].ne("")

    missing_text_count = int((~prepared["_has_usable_text"]).sum())
    duplicate_article_ids = int(prepared["_article_id"].duplicated().sum())

    logger.info("Rows with usable full text: %s", int(prepared["_has_usable_text"].sum()))
    logger.info("Rows missing full text: %s", missing_text_count)
    logger.info("Duplicate article IDs: %s", duplicate_article_ids)
    logger.info(
        "Rows with repaired model text: %s",
        int(prepared["_text_encoding_repaired"].sum()),
    )

    prepared["_run_chunk_number"] = (
        prepared["_source_row_number"] // RUN_CHUNK_SIZE
    ) + 1

    total_chunks = int(math.ceil(len(prepared) / RUN_CHUNK_SIZE))
    logger.info("Total 700-row run chunks: %s", total_chunks)

    if CHUNK_NUMBER < 1 or CHUNK_NUMBER > total_chunks:
        raise ValueError(
            f"CHUNK_NUMBER must be between 1 and {total_chunks}; got {CHUNK_NUMBER}"
        )

    return prepared


df_all = load_and_prepare_dataset(INPUT_CSV)
TOTAL_RUN_CHUNKS = int(math.ceil(len(df_all) / RUN_CHUNK_SIZE))

chunk_manifest = (
    df_all.groupby("_run_chunk_number", as_index=False)
    .agg(
        first_source_row=("_source_row_number", "min"),
        last_source_row=("_source_row_number", "max"),
        rows=("_source_row_number", "size"),
        usable_full_text_rows=("_has_usable_text", "sum"),
    )
)
chunk_manifest.to_csv(WORKING_DIR / "chunk_manifest.csv", index=False)
display(chunk_manifest)

if WRITE_ALL_RAW_CHUNKS:
    for chunk_number, chunk_frame in df_all.groupby("_run_chunk_number", sort=True):
        raw_chunk_path = RAW_CHUNKS_DIR / f"raw_chunk_{int(chunk_number):03d}.csv"
        source_columns = [
            column for column in chunk_frame.columns
            if not column.startswith("_")
        ]
        chunk_frame[source_columns].to_csv(raw_chunk_path, index=False)
    logger.info("Raw 700-row chunk files written to %s", RAW_CHUNKS_DIR)

df_chunk = df_all.loc[
    df_all["_run_chunk_number"].eq(CHUNK_NUMBER)
].copy().reset_index(drop=True)

logger.info(
    "Selected chunk %03d: %s rows, %s usable full-text rows",
    CHUNK_NUMBER,
    len(df_chunk),
    int(df_chunk["_has_usable_text"].sum()),
)

display(
    df_chunk[
        [
            "_source_row_number",
            "_article_id",
            "_model_date",
            "_model_headline",
            "_text_char_count",
            "_text_word_count",
            "_has_usable_text",
        ]
    ].head()
)

2026-07-28T05:59:54 | INFO | CSV loaded with encoding: utf-8
2026-07-28T05:59:54 | INFO | Source rows: 6542
2026-07-28T05:59:54 | INFO | Source columns: ['article_id', 'collection', 'headline', 'retrieved_title', 'date', 'retrieved_publish_date', 'source_name', 'source_domain', 'source_url', 'final_url', 'preferred_url', 'canonical_url', 'retrieved_text', 'calculated_text_length', 'calculated_word_count', 'retrieved_text_hash', 'normalized_text_hash', 'retrieval_method', 'retrieved_at_utc', 'duplicate_group_id', 'duplicate_group_size', 'quality_score', 'deduplication_status', 'input_file', 'input_row_number', 'url', 'retrieval_status', 'error_type', 'error_message', 'retrieved_text_len', 'retrieved_word_count', 'retrieved_authors', 'retrieved_top_image', 'retrieved_meta_description', 'retrieved_meta_lang', 'retrieved_keywords', 'retrieved_summary', 'source_file', 'source_dataset', 'headline_clean', 'headline_hash', 'legacy_score', 'legacy_themes', 'legacy_dimension', 'input_rows_before

,_run_chunk_number,first_source_row,last_source_row,rows,usable_full_text_rows
0,1,0,699,700,700
1,2,700,1399,700,700
2,3,1400,2099,700,700
3,4,2100,2799,700,700
4,5,2800,3499,700,700
5,6,3500,4199,700,700
6,7,4200,4899,700,700
7,8,4900,5599,700,700
8,9,5600,6299,700,700
9,10,6300,6541,242,242


2026-07-28T06:00:10 | INFO | Raw 700-row chunk files written to /kaggle/working/shipping_disruption_scoring_gemma2_9b/raw_chunks
2026-07-28T06:00:10 | INFO | Selected chunk 010: 242 rows, 242 usable full-text rows


,_source_row_number,_article_id,_model_date,_model_headline,_text_char_count,_text_word_count,_has_usable_text
0,6300,llm_news_0010727,2025-10-24T15:02:35+00:00,U.S. container ports putting pandemic in the r...,3096,472,True
1,6301,llm_news_0009397,,China Gains Foothold in Thailand as Cosco Inve...,2034,330,True
2,6302,llm_news_0009398,2024-10-25T07:27:49+00:00,Dali owners settle lawsuit with US Government,2507,415,True
3,6303,llm_news_0009400,2024-10-25T08:07:45+05:30,How J N Port is tackling empty container yards...,7395,1223,True
4,6304,llm_news_0009402,2024-10-26T00:00:00,Ministers urged to cut ties with P&O Ferries o...,4876,788,True


## 7. Load the Gemma 2 tokenizer and model

Gemma 2 9B IT is loaded through `AutoModelForCausalLM` in four-bit NF4
mode. This retains the same quantized-inference strategy used for the
earlier Q1 runs while keeping the 9B model practical on a Kaggle T4.


In [7]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Enable a GPU accelerator in Kaggle before "
        "loading the model."
    )

logger.info("Loading tokenizer from: %s", MODEL_SOURCE)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_SOURCE,
)

model_kwargs: dict[str, Any] = {
    "device_map": "auto",
    "low_cpu_mem_usage": True,
}

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = quantization_config
    model_kwargs["torch_dtype"] = torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float16

logger.info("Loading Gemma 2 model from: %s", MODEL_SOURCE)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_SOURCE,
    **model_kwargs,
)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

MODEL_INPUT_DEVICE = next(model.parameters()).device
MODEL_CONTEXT_LENGTH = int(
    getattr(model.config, "max_position_embeddings", 8192)
)

logger.info("Model loaded; prompt input device: %s", MODEL_INPUT_DEVICE)
logger.info("Model context length: %s", MODEL_CONTEXT_LENGTH)
logger.info("Canonical model name recorded in outputs: %s", MODEL_NAME)


2026-07-28T06:00:10 | INFO | Loading tokenizer from: /kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2
2026-07-28T06:00:11 | INFO | Loading Gemma 2 model from: /kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2026-07-28T06:02:51 | INFO | Model loaded; prompt input device: cuda:0
2026-07-28T06:02:51 | INFO | Model context length: 8192
2026-07-28T06:02:51 | INFO | Canonical model name recorded in outputs: /kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2


## 8. Full-text windowing and model inference

Most articles will use one window. Articles longer than the configured token limit are split into overlapping windows. Theme scores are then aggregated by the maximum severity found anywhere in the article, while retaining the evidence associated with that maximum.

In [8]:
def split_text_into_token_windows(text: str) -> list[dict[str, Any]]:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    total_tokens = len(token_ids)

    if total_tokens == 0:
        return []

    if total_tokens <= MAX_TEXT_TOKENS_PER_WINDOW:
        return [
            {
                "window_number": 1,
                "window_count": 1,
                "start_token": 0,
                "end_token": total_tokens,
                "text": text,
            }
        ]

    step = MAX_TEXT_TOKENS_PER_WINDOW - WINDOW_TOKEN_OVERLAP
    if step <= 0:
        raise ValueError("WINDOW_TOKEN_OVERLAP must be smaller than the window size")

    windows: list[dict[str, Any]] = []
    starts = list(range(0, total_tokens, step))

    for window_number, start in enumerate(starts, start=1):
        end = min(start + MAX_TEXT_TOKENS_PER_WINDOW, total_tokens)
        window_text = tokenizer.decode(
            token_ids[start:end],
            skip_special_tokens=True,
        )
        windows.append(
            {
                "window_number": window_number,
                "window_count": 0,
                "start_token": start,
                "end_token": end,
                "text": window_text,
            }
        )
        if end >= total_tokens:
            break

    for window in windows:
        window["window_count"] = len(windows)

    return windows


def build_user_prompt(
    article_id: str,
    headline: str,
    date_text: str,
    window: dict[str, Any],
) -> str:
    return f'''
ARTICLE ID: {article_id}
PUBLICATION DATE: {date_text or "not supplied"}
HEADLINE: {headline or "not supplied"}
TEXT WINDOW: {window["window_number"]} of {window["window_count"]}

Score this article window using the required seven-theme 0-4 taxonomy.

ARTICLE TEXT:
{window["text"]}
'''.strip()


def generate_raw_response(user_prompt: str) -> tuple[str, int, int]:
    # Gemma 2 IT uses user/model turns and does not use a separate system
    # role in its official chat template. The unchanged Q1 system rubric is
    # therefore prepended to the unchanged article prompt.
    combined_prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"
    messages = [
        {
            "role": "user",
            "content": combined_prompt,
        }
    ]

    rendered_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        rendered_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )
    inputs = {
        name: tensor.to(MODEL_INPUT_DEVICE)
        for name, tensor in inputs.items()
    }

    input_token_count = int(inputs["input_ids"].shape[1])
    available_generation_tokens = MODEL_CONTEXT_LENGTH - input_token_count

    if available_generation_tokens < 64:
        raise ValueError(
            "Prompt leaves fewer than 64 tokens for generation: "
            f"input={input_token_count}, context={MODEL_CONTEXT_LENGTH}. "
            "Reduce MAX_TEXT_TOKENS_PER_WINDOW if this occurs."
        )

    generation_max_new_tokens = min(
        MAX_NEW_TOKENS,
        available_generation_tokens,
    )

    if generation_max_new_tokens < MAX_NEW_TOKENS:
        logger.info(
            "Generation cap reduced from %s to %s tokens to remain within "
            "Gemma 2 context length",
            MAX_NEW_TOKENS,
            generation_max_new_tokens,
        )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=generation_max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    prompt_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0, prompt_length:]
    generated_token_count = int(generated_ids.shape[0])
    raw_response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    del inputs, outputs, generated_ids
    torch.cuda.empty_cache()

    return raw_response, input_token_count, generated_token_count


def classify_one_window(
    row: pd.Series,
    window: dict[str, Any],
) -> dict[str, Any]:
    article_id = str(row["_article_id"])
    row_uid = str(row["_row_uid"])
    prompt = build_user_prompt(
        article_id=article_id,
        headline=str(row["_model_headline"]),
        date_text=str(row["_model_date"]),
        window=window,
    )

    last_error: Exception | None = None

    for attempt in range(1, MAX_RETRIES + 2):
        started_at = utc_now_iso()
        start_time = time.time()
        raw_response = ""
        input_token_count = 0
        generated_token_count = 0

        try:
            raw_response, input_token_count, generated_token_count = (
                generate_raw_response(prompt)
            )
            parsed_payload = extract_json_object(raw_response)
            validated = validate_model_result(parsed_payload)
            elapsed_seconds = time.time() - start_time

            if validated.get("overall_reasoning_generated", False):
                logger.info(
                    "Row %s, window %s/%s: model omitted overall_reasoning; "
                    "deterministic fallback generated without retry",
                    row_uid,
                    window["window_number"],
                    window["window_count"],
                )

            audit_payload = {
                "event": "window_classification",
                "status": "success",
                "started_at_utc": started_at,
                "finished_at_utc": utc_now_iso(),
                "elapsed_seconds": elapsed_seconds,
                "chunk_number": CHUNK_NUMBER,
                "source_row_number": int(row["_source_row_number"]),
                "row_uid": row_uid,
                "article_id": article_id,
                "attempt": attempt,
                "window_number": int(window["window_number"]),
                "window_count": int(window["window_count"]),
                "window_start_token": int(window["start_token"]),
                "window_end_token": int(window["end_token"]),
                "input_token_count": input_token_count,
                "generated_token_count": generated_token_count,
                "validated_result": validated,
            }
            if SAVE_RAW_MODEL_RESPONSE:
                audit_payload["raw_model_response"] = raw_response

            append_jsonl(AUDIT_JSONL_FILE, audit_payload)

            return {
                **validated,
                "_window_number": int(window["window_number"]),
                "_window_count": int(window["window_count"]),
                "_window_start_token": int(window["start_token"]),
                "_window_end_token": int(window["end_token"]),
                "_input_token_count": input_token_count,
                "_generated_token_count": generated_token_count,
                "_attempts_used": attempt,
                "_elapsed_seconds": elapsed_seconds,
            }

        except Exception as error:
            last_error = error
            elapsed_seconds = time.time() - start_time

            audit_payload = {
                "event": "window_classification",
                "status": "failed_attempt",
                "started_at_utc": started_at,
                "finished_at_utc": utc_now_iso(),
                "elapsed_seconds": elapsed_seconds,
                "chunk_number": CHUNK_NUMBER,
                "source_row_number": int(row["_source_row_number"]),
                "row_uid": row_uid,
                "article_id": article_id,
                "attempt": attempt,
                "window_number": int(window["window_number"]),
                "window_count": int(window["window_count"]),
                "input_token_count": input_token_count,
                "generated_token_count": generated_token_count,
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
            if SAVE_RAW_MODEL_RESPONSE and raw_response:
                audit_payload["raw_model_response"] = raw_response

            append_jsonl(AUDIT_JSONL_FILE, audit_payload)

            logger.warning(
                "Row %s, window %s/%s, attempt %s failed: %s: %s",
                row_uid,
                window["window_number"],
                window["window_count"],
                attempt,
                type(error).__name__,
                str(error)[:300],
            )

            torch.cuda.empty_cache()
            gc.collect()

            if attempt <= MAX_RETRIES:
                time.sleep(RETRY_DELAY_SECONDS * attempt)

    raise RuntimeError(
        f"Window classification failed after {MAX_RETRIES + 1} attempts"
    ) from last_error


def aggregate_window_results(
    window_results: list[dict[str, Any]],
) -> dict[str, Any]:
    if not window_results:
        raise ValueError("No successful window results to aggregate")

    aggregate_scores: dict[str, int] = {}
    aggregate_evidence: dict[str, str] = {}

    for model_key, score_column in THEME_KEYS.items():
        best_result = max(
            window_results,
            key=lambda result: int(result[score_column]),
        )
        aggregate_scores[score_column] = int(best_result[score_column])
        aggregate_evidence[f"{model_key}_evidence"] = str(
            best_result[f"{model_key}_evidence"]
        )

    total_input_tokens = sum(
        int(result["_input_token_count"]) for result in window_results
    )
    total_generated_tokens = sum(
        int(result["_generated_token_count"]) for result in window_results
    )
    total_elapsed_seconds = sum(
        float(result["_elapsed_seconds"]) for result in window_results
    )
    attempts_used = sum(
        int(result["_attempts_used"]) for result in window_results
    )

    reasoning_parts = [
        str(result["overall_reasoning"])
        for result in window_results
        if str(result["overall_reasoning"]).strip()
    ]
    combined_reasoning = " | ".join(dict.fromkeys(reasoning_parts))
    if len(combined_reasoning) > 1500:
        combined_reasoning = combined_reasoning[:1497] + "..."

    reasoning_fallback_window_count = sum(
        bool(result.get("overall_reasoning_generated", False))
        for result in window_results
    )

    return {
        **aggregate_scores,
        **aggregate_evidence,
        **derived_summary(aggregate_scores),
        "overall_reasoning": combined_reasoning,
        "overall_reasoning_generated": reasoning_fallback_window_count > 0,
        "reasoning_fallback_window_count": reasoning_fallback_window_count,
        "text_window_count": len(window_results),
        "total_input_tokens": total_input_tokens,
        "total_generated_tokens": total_generated_tokens,
        "total_model_attempts": attempts_used,
        "model_elapsed_seconds": total_elapsed_seconds,
    }


def classify_full_article(row: pd.Series) -> dict[str, Any]:
    windows = split_text_into_token_windows(str(row["_model_text"]))
    if not windows:
        raise ValueError("Article has no usable tokenized text")

    logger.info(
        "Scoring row_uid=%s article_id=%s windows=%s words=%s",
        row["_row_uid"],
        row["_article_id"],
        len(windows),
        row["_text_word_count"],
    )

    window_results = [
        classify_one_window(row, window)
        for window in windows
    ]

    article_result = aggregate_window_results(window_results)

    append_jsonl(
        AUDIT_JSONL_FILE,
        {
            "event": "article_aggregation",
            "status": "success",
            "timestamp_utc": utc_now_iso(),
            "chunk_number": CHUNK_NUMBER,
            "source_row_number": int(row["_source_row_number"]),
            "row_uid": str(row["_row_uid"]),
            "article_id": str(row["_article_id"]),
            "window_count": len(windows),
            "aggregated_result": article_result,
        },
    )
    return article_result

## 9. Smoke test

Use a small number of rows before starting the complete 700-row chunk.

In [9]:
SMOKE_TEST_ROWS = 3

smoke_source = (
    df_chunk.loc[df_chunk["_has_usable_text"]]
    .head(SMOKE_TEST_ROWS)
)

smoke_results: list[dict[str, Any]] = []

for smoke_position, (_, smoke_row) in enumerate(
    smoke_source.iterrows(),
    start=1,
):
    print("\n" + "=" * 100, flush=True)
    print(
        f"SMOKE TEST ARTICLE {smoke_position}/{len(smoke_source)}",
        flush=True,
    )
    print("=" * 100, flush=True)

    print(
        f"Article ID : {smoke_row['_article_id']}",
        flush=True,
    )
    print(
        f"Headline   : {smoke_row['_model_headline']}",
        flush=True,
    )
    print(
        f"Word count : {smoke_row['_text_word_count']}",
        flush=True,
    )
    print("-" * 100, flush=True)

    result = classify_full_article(smoke_row)

    print("\nTHEME SCORES AND EVIDENCE", flush=True)
    print("-" * 100, flush=True)

    for model_key, score_column in THEME_KEYS.items():
        evidence_column = f"{model_key}_evidence"

        print(
            f"{score_column}: {result[score_column]}",
            flush=True,
        )
        print(
            f"Evidence: {result[evidence_column] or 'No evidence'}",
            flush=True,
        )
        print("-" * 50, flush=True)

    print("\nAGGREGATE RESULTS", flush=True)
    print("-" * 100, flush=True)

    print(
        f"Shipping dimension max  : "
        f"{result['shipping_dimension_max']}",
        flush=True,
    )
    print(
        f"Shipping dimension mean : "
        f"{result['shipping_dimension_mean']:.3f}",
        flush=True,
    )
    print(
        f"Shipping dimension sum  : "
        f"{result['shipping_dimension_sum']}",
        flush=True,
    )

    print(
        f"Economic dimension max  : "
        f"{result['economic_dimension_max']}",
        flush=True,
    )
    print(
        f"Economic dimension mean : "
        f"{result['economic_dimension_mean']:.3f}",
        flush=True,
    )
    print(
        f"Economic dimension sum  : "
        f"{result['economic_dimension_sum']}",
        flush=True,
    )

    print(
        f"Overall disruption max  : "
        f"{result['overall_disruption_max']}",
        flush=True,
    )
    print(
        f"Overall disruption mean : "
        f"{result['overall_disruption_mean']:.3f}",
        flush=True,
    )
    print(
        f"Overall disruption sum  : "
        f"{result['overall_disruption_sum']}",
        flush=True,
    )
    print(
        f"Active theme count      : "
        f"{result['active_theme_count']}",
        flush=True,
    )

    print("\nINFERENCE INFORMATION", flush=True)
    print("-" * 100, flush=True)

    print(
        f"Text windows             : "
        f"{result['text_window_count']}",
        flush=True,
    )
    print(
        f"Input tokens             : "
        f"{result['total_input_tokens']}",
        flush=True,
    )
    print(
        f"Generated tokens         : "
        f"{result['total_generated_tokens']}",
        flush=True,
    )
    print(
        f"Total model attempts     : "
        f"{result['total_model_attempts']}",
        flush=True,
    )
    print(
        f"Inference time           : "
        f"{result['model_elapsed_seconds']:.2f} seconds",
        flush=True,
    )
    print(
        f"Reasoning fallback used  : "
        f"{result['overall_reasoning_generated']}",
        flush=True,
    )
    print(
        f"Fallback window count    : "
        f"{result['reasoning_fallback_window_count']}",
        flush=True,
    )

    print("\nOVERALL REASONING", flush=True)
    print("-" * 100, flush=True)
    print(result["overall_reasoning"], flush=True)

    smoke_results.append(
        {
            "article_id": smoke_row["_article_id"],
            "headline": smoke_row["_model_headline"],
            **{
                column: result[column]
                for column in THEME_COLUMNS
            },
            "shipping_dimension_max": result[
                "shipping_dimension_max"
            ],
            "economic_dimension_max": result[
                "economic_dimension_max"
            ],
            "overall_disruption_max": result[
                "overall_disruption_max"
            ],
            "active_theme_count": result[
                "active_theme_count"
            ],
            "text_window_count": result[
                "text_window_count"
            ],
            "total_input_tokens": result[
                "total_input_tokens"
            ],
            "total_generated_tokens": result[
                "total_generated_tokens"
            ],
            "total_model_attempts": result[
                "total_model_attempts"
            ],
            "model_elapsed_seconds": result[
                "model_elapsed_seconds"
            ],
            "overall_reasoning_generated": result[
                "overall_reasoning_generated"
            ],
            "overall_reasoning": result[
                "overall_reasoning"
            ],
        }
    )

    print(
        f"\nFinished article {smoke_position}/"
        f"{len(smoke_source)}",
        flush=True,
    )

print("\n" + "=" * 100)
print("SMOKE TEST SUMMARY")
print("=" * 100)

smoke_results_df = pd.DataFrame(smoke_results)
display(smoke_results_df)


SMOKE TEST ARTICLE 1/3
Article ID : llm_news_0010727
Headline   : U.S. container ports putting pandemic in the rearview
Word count : 472
----------------------------------------------------------------------------------------------------


2026-07-28T06:02:51 | INFO | Scoring row_uid=llm_news_0010727__6300 article_id=llm_news_0010727 windows=1 words=472



THEME SCORES AND EVIDENCE
----------------------------------------------------------------------------------------------------
port_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
global_container_movement_disruption_score: 1
Evidence: The article mentions 'fluctuating demand and shifting trade routes as carriers minimize exposure to U.S. port fees on Chinese-built ships' as challenges in the global shipping industry.
--------------------------------------------------
freight_cost_disruption_score: 1
Evidence: The article states that 'freight rate fluctuations and changing geopolitical pressures' are impacting shipping logistics and cost structures.
--------------------------------------------------
supply_disruption_score: 1
Evidence: The article notes 'broader challenges in the global shipping industry, including fluctuating demand and shifting trade routes as carriers minimize exposure to U.S. port fees on Chinese-built ships.'
---------

2026-07-28T06:03:45 | INFO | Scoring row_uid=llm_news_0009397__6301 article_id=llm_news_0009397 windows=1 words=330



THEME SCORES AND EVIDENCE
----------------------------------------------------------------------------------------------------
port_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
global_container_movement_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
freight_cost_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
supply_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
production_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
covid19_related_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
demand_disruption_score: 0
Evidence: No evidence
--------------------------------------------------

AGGREGATE RESULTS
----------------------------------------------------------------------------------------------------
Shipping dimensi

2026-07-28T06:04:12 | INFO | Scoring row_uid=llm_news_0009398__6302 article_id=llm_news_0009398 windows=1 words=415



THEME SCORES AND EVIDENCE
----------------------------------------------------------------------------------------------------
port_disruption_score: 2
Evidence: The Dali struck the bridge, obstructing the channel and blocking all shipping in and out of the port. Temporary channels were set up, and the port reopened on June 10th.
--------------------------------------------------
global_container_movement_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
freight_cost_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
supply_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
production_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
covid19_related_disruption_score: 0
Evidence: No evidence
--------------------------------------------------
demand_disruption_score: 0
Evidence: No evidence
-----------------------------

,article_id,headline,port_disruption_score,global_container_movement_disruption_score,freight_cost_disruption_score,supply_disruption_score,production_disruption_score,covid19_related_disruption_score,demand_disruption_score,shipping_dimension_max,economic_dimension_max,overall_disruption_max,active_theme_count,text_window_count,total_input_tokens,total_generated_tokens,total_model_attempts,model_elapsed_seconds,overall_reasoning_generated,overall_reasoning
0,llm_news_0010727,U.S. container ports putting pandemic in the r...,0,1,1,1,0,0,1,1,1,1,4,1,1932,378,1,53.526308,False,The article primarily discusses trends in cont...
1,llm_news_0009397,China Gains Foothold in Thailand as Cosco Inve...,0,0,0,0,0,0,0,0,0,0,0,1,1572,203,1,27.017875,False,No taxonomy-defined disruption was identified ...
2,llm_news_0009398,Dali owners settle lawsuit with US Government,2,0,0,0,0,0,0,2,0,2,1,1,1654,266,1,35.156000,False,The article describes a maritime accident that...


## 10. Resumable scoring of the selected 700-row chunk

The checkpoint includes successful, failed, and missing-text records. On rerun, only successful records and explicit missing-text records are treated as complete; failed inference records are retried.

In [10]:
EVIDENCE_COLUMNS = [
    "port_disruption_evidence",
    "global_container_movement_disruption_evidence",
    "freight_cost_disruption_evidence",
    "supply_disruption_evidence",
    "production_disruption_evidence",
    "covid19_related_disruption_evidence",
    "demand_disruption_evidence",
]

DERIVED_COLUMNS = [
    "shipping_dimension_max",
    "shipping_dimension_mean",
    "shipping_dimension_sum",
    "economic_dimension_max",
    "economic_dimension_mean",
    "economic_dimension_sum",
    "overall_disruption_max",
    "overall_disruption_mean",
    "overall_disruption_sum",
    "active_theme_count",
]

PROCESS_COLUMNS = [
    "scoring_status",
    "scoring_error_type",
    "scoring_error_message",
    "overall_reasoning_generated",
    "reasoning_fallback_window_count",
    "scored_at_utc",
    "model_name",
    "taxonomy_version",
    "severity_scale",
    "text_window_count",
    "total_input_tokens",
    "total_generated_tokens",
    "total_model_attempts",
    "model_elapsed_seconds",
    "model_text_char_count",
    "model_text_word_count",
    "text_encoding_repaired",
    "run_chunk_number",
    "source_row_number",
    "row_uid",
]

ALL_APPENDED_COLUMNS = (
    THEME_COLUMNS
    + EVIDENCE_COLUMNS
    + DERIVED_COLUMNS
    + ["overall_reasoning"]
    + PROCESS_COLUMNS
)


def atomic_write_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def load_existing_results(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()

    existing = pd.read_csv(path, low_memory=False)
    required = {"row_uid", "scoring_status"}
    missing = required - set(existing.columns)
    if missing:
        logger.warning(
            "Ignoring incompatible checkpoint; missing columns: %s",
            sorted(missing),
        )
        return pd.DataFrame()

    existing = existing.drop_duplicates(subset="row_uid", keep="last")
    logger.info("Loaded checkpoint rows: %s", len(existing))
    return existing


def source_row_for_output(row: pd.Series) -> dict[str, Any]:
    return {
        column: row[column]
        for column in df_chunk.columns
        if not column.startswith("_")
    }


def build_success_output(row: pd.Series, result: dict[str, Any]) -> dict[str, Any]:
    return {
        **source_row_for_output(row),
        **{column: result[column] for column in THEME_COLUMNS},
        **{column: result[column] for column in EVIDENCE_COLUMNS},
        **{column: result[column] for column in DERIVED_COLUMNS},
        "overall_reasoning": result["overall_reasoning"],
        "scoring_status": "success",
        "scoring_error_type": "",
        "scoring_error_message": "",
        "overall_reasoning_generated": bool(
            result.get("overall_reasoning_generated", False)
        ),
        "reasoning_fallback_window_count": int(
            result.get("reasoning_fallback_window_count", 0)
        ),
        "scored_at_utc": utc_now_iso(),
        "model_name": MODEL_NAME,
        "taxonomy_version": "Munim_2025_2_dimensions_7_themes_17_codes",
        "severity_scale": "0-4",
        "text_window_count": result["text_window_count"],
        "total_input_tokens": result["total_input_tokens"],
        "total_generated_tokens": result["total_generated_tokens"],
        "total_model_attempts": result["total_model_attempts"],
        "model_elapsed_seconds": result["model_elapsed_seconds"],
        "model_text_char_count": int(row["_text_char_count"]),
        "model_text_word_count": int(row["_text_word_count"]),
        "text_encoding_repaired": bool(row["_text_encoding_repaired"]),
        "run_chunk_number": int(row["_run_chunk_number"]),
        "source_row_number": int(row["_source_row_number"]),
        "row_uid": str(row["_row_uid"]),
    }


def build_non_success_output(
    row: pd.Series,
    status: str,
    error: Exception | None = None,
) -> dict[str, Any]:
    output = source_row_for_output(row)

    for column in THEME_COLUMNS + DERIVED_COLUMNS:
        output[column] = np.nan
    for column in EVIDENCE_COLUMNS:
        output[column] = ""
    output["overall_reasoning"] = ""

    output.update(
        {
            "scoring_status": status,
            "scoring_error_type": type(error).__name__ if error else "",
            "scoring_error_message": str(error) if error else "",
            "overall_reasoning_generated": False,
            "reasoning_fallback_window_count": 0,
            "scored_at_utc": utc_now_iso(),
            "model_name": MODEL_NAME,
            "taxonomy_version": "Munim_2025_2_dimensions_7_themes_17_codes",
            "severity_scale": "0-4",
            "text_window_count": 0,
            "total_input_tokens": 0,
            "total_generated_tokens": 0,
            "total_model_attempts": 0,
            "model_elapsed_seconds": 0.0,
            "model_text_char_count": int(row["_text_char_count"]),
            "model_text_word_count": int(row["_text_word_count"]),
            "text_encoding_repaired": bool(row["_text_encoding_repaired"]),
            "run_chunk_number": int(row["_run_chunk_number"]),
            "source_row_number": int(row["_source_row_number"]),
            "row_uid": str(row["_row_uid"]),
        }
    )
    return output


def save_current_results(records: list[dict[str, Any]]) -> pd.DataFrame:
    frame = pd.DataFrame(records)
    frame = frame.drop_duplicates(subset="row_uid", keep="last")
    frame = frame.sort_values("source_row_number", kind="stable").reset_index(drop=True)
    atomic_write_csv(frame, CHECKPOINT_FILE)
    return frame


def run_selected_chunk() -> pd.DataFrame:
    existing = load_existing_results(CHECKPOINT_FILE)
    records = existing.to_dict("records") if not existing.empty else []

    completed_statuses = {"success", "missing_text"}
    completed_uids = set(
        existing.loc[
            existing["scoring_status"].isin(completed_statuses),
            "row_uid",
        ].astype(str)
    ) if not existing.empty else set()

    pending = df_chunk.loc[
        ~df_chunk["_row_uid"].astype(str).isin(completed_uids)
    ].copy()

    logger.info("Chunk rows: %s", len(df_chunk))
    logger.info("Completed rows retained: %s", len(completed_uids))
    logger.info("Rows pending or retryable: %s", len(pending))

    run_started = time.time()

    for position, (_, row) in enumerate(pending.iterrows(), start=1):
        row_uid = str(row["_row_uid"])

        # Remove any older failed record before appending the retry result.
        records = [
            record for record in records
            if str(record.get("row_uid")) != row_uid
        ]

        if not bool(row["_has_usable_text"]):
            output = build_non_success_output(
                row,
                status="missing_text",
                error=None,
            )
            records.append(output)
            append_jsonl(
                AUDIT_JSONL_FILE,
                {
                    "event": "article_skipped",
                    "status": "missing_text",
                    "timestamp_utc": utc_now_iso(),
                    "chunk_number": CHUNK_NUMBER,
                    "source_row_number": int(row["_source_row_number"]),
                    "row_uid": row_uid,
                    "article_id": str(row["_article_id"]),
                },
            )
            logger.warning("Skipped missing text: %s", row_uid)

        else:
            try:
                article_result = classify_full_article(row)
                output = build_success_output(row, article_result)
                records.append(output)
                logger.info(
                    "Completed %s/%s pending rows | row_uid=%s | overall_max=%s",
                    position,
                    len(pending),
                    row_uid,
                    article_result["overall_disruption_max"],
                )

            except Exception as error:
                output = build_non_success_output(
                    row,
                    status="failed",
                    error=error,
                )
                records.append(output)
                append_jsonl(
                    AUDIT_JSONL_FILE,
                    {
                        "event": "article_classification",
                        "status": "failed",
                        "timestamp_utc": utc_now_iso(),
                        "chunk_number": CHUNK_NUMBER,
                        "source_row_number": int(row["_source_row_number"]),
                        "row_uid": row_uid,
                        "article_id": str(row["_article_id"]),
                        "error_type": type(error).__name__,
                        "error_message": str(error),
                    },
                )
                logger.exception("Article failed: %s", row_uid)

        if position % CHECKPOINT_EVERY_ARTICLES == 0:
            current = save_current_results(records)
            logger.info(
                "Checkpoint saved after %s pending rows; stored rows=%s",
                position,
                len(current),
            )

    final_frame = save_current_results(records)
    atomic_write_csv(final_frame, FINAL_OUTPUT_FILE)

    failures = final_frame.loc[
        final_frame["scoring_status"].eq("failed")
    ].copy()
    atomic_write_csv(failures, FAILURES_FILE)

    logger.info("Chunk run completed in %.2f minutes", (time.time() - run_started) / 60)
    logger.info("Final rows: %s", len(final_frame))
    logger.info(
        "Successful: %s",
        int(final_frame["scoring_status"].eq("success").sum()),
    )
    logger.info(
        "Missing text: %s",
        int(final_frame["scoring_status"].eq("missing_text").sum()),
    )
    logger.info("Failed: %s", len(failures))
    logger.info("Final output: %s", FINAL_OUTPUT_FILE)

    return final_frame


df_scored_chunk = run_selected_chunk()
display(df_scored_chunk.head())

2026-07-28T06:04:47 | INFO | Chunk rows: 242
2026-07-28T06:04:47 | INFO | Completed rows retained: 0
2026-07-28T06:04:47 | INFO | Rows pending or retryable: 242
2026-07-28T06:04:47 | INFO | Scoring row_uid=llm_news_0010727__6300 article_id=llm_news_0010727 windows=1 words=472
2026-07-28T06:05:41 | INFO | Completed 1/242 pending rows | row_uid=llm_news_0010727__6300 | overall_max=1
2026-07-28T06:05:41 | INFO | Scoring row_uid=llm_news_0009397__6301 article_id=llm_news_0009397 windows=1 words=330
2026-07-28T06:06:08 | INFO | Completed 2/242 pending rows | row_uid=llm_news_0009397__6301 | overall_max=0
2026-07-28T06:06:08 | INFO | Scoring row_uid=llm_news_0009398__6302 article_id=llm_news_0009398 windows=1 words=415
2026-07-28T06:06:43 | INFO | Completed 3/242 pending rows | row_uid=llm_news_0009398__6302 | overall_max=2
2026-07-28T06:06:43 | INFO | Scoring row_uid=llm_news_0009400__6303 article_id=llm_news_0009400 windows=1 words=1223
2026-07-28T06:07:40 | INFO | Completed 4/242 pending 

,article_id,collection,headline,retrieved_title,date,retrieved_publish_date,source_name,source_domain,source_url,final_url,...,total_input_tokens,total_generated_tokens,total_model_attempts,model_elapsed_seconds,model_text_char_count,model_text_word_count,text_encoding_repaired,run_chunk_number,source_row_number,row_uid
0,llm_news_0010727,current,U.S. container ports putting pandemic in the r...,U.S. container ports putting pandemic in the r...,"Oct 24, 2025",2025-10-24T15:02:35+00:00,FreightWaves,freightwaves.com,https://www.freightwaves.com/news/u-s-containe...,https://www.freightwaves.com/news/u-s-containe...,...,1932,378,1,53.741301,3096,472,True,10,6300,llm_news_0010727__6300
1,llm_news_0009397,current,China Gains Foothold in Thailand as Cosco Inve...,China Gains Foothold in Thailand as Cosco Inve...,"Oct 25, 2024",NaN,The Maritime Executive,maritime-executive.com,https://maritime-executive.com/article/china-g...,https://maritime-executive.com/article/china-g...,...,1572,203,1,26.645338,2034,330,True,10,6301,llm_news_0009397__6301
2,llm_news_0009398,current,Dali owners settle lawsuit with US Government,Dali owners settle lawsuit with US Government,"Oct 25, 2024",2024-10-25T07:27:49+00:00,Seatrade Maritime News,seatrade-maritime.com,https://www.seatrade-maritime.com/accidents/da...,https://www.seatrade-maritime.com/accidents/da...,...,1654,266,1,35.118427,2507,415,True,10,6302,llm_news_0009398__6302
3,llm_news_0009400,current,How J N Port is tackling empty container yards...,How J N Port is tackling empty container yards...,"Oct 25, 2024",2024-10-25T08:07:45+05:30,ET Infra,infra.economictimes.indiatimes.com,https://infra.economictimes.indiatimes.com/new...,https://infra.economictimes.indiatimes.com/new...,...,2632,334,1,56.964880,7395,1223,True,10,6303,llm_news_0009400__6303
4,llm_news_0009402,current,Ministers urged to cut ties with P&O Ferries o...,Ministers urged to cut ties with P&O Ferries o...,"Oct 26, 2024",2024-10-26T00:00:00,The Guardian,theguardian.com,https://www.theguardian.com/business/2024/oct/...,https://www.theguardian.com/business/2024/oct/...,...,2184,282,1,43.564882,4876,788,True,10,6304,llm_news_0009402__6304


## 11. Quality-control summary

Review failure counts, score distributions, theme correlations, and a sample of explanations before accepting the chunk.

In [11]:
print("=" * 80)
print(f"CHUNK {CHUNK_NUMBER:03d} QUALITY-CONTROL SUMMARY")
print("=" * 80)
print(df_scored_chunk["scoring_status"].value_counts(dropna=False))

successful = df_scored_chunk.loc[
    df_scored_chunk["scoring_status"].eq("success")
].copy()

if successful.empty:
    print("\nNo successful rows are available for score summaries.")
else:
    print(
        "\nRows using generated overall reasoning:",
        int(successful["overall_reasoning_generated"].fillna(False).sum()),
    )

    print("\nTheme score distributions")
    for column in THEME_COLUMNS:
        print(f"\n{column}")
        print(
            successful[column]
            .value_counts(dropna=False)
            .sort_index()
            .to_string()
        )

    print("\nDescriptive statistics")
    display(successful[THEME_COLUMNS + DERIVED_COLUMNS].describe().T)

    print("\nTheme score correlation")
    display(successful[THEME_COLUMNS].corr())

    explanation_columns = [
        column for column in [
            ID_COLUMN,
            HEADLINE_COLUMN,
            DATE_COLUMN,
        ]
        if column is not None and column in successful.columns
    ]
    explanation_columns += THEME_COLUMNS + ["overall_reasoning"]
    display(successful[explanation_columns].sample(min(10, len(successful)), random_state=42))

CHUNK 010 QUALITY-CONTROL SUMMARY
scoring_status
success    242
Name: count, dtype: int64

Rows using generated overall reasoning: 0

Theme score distributions

port_disruption_score
port_disruption_score
0    179
1     11
2     48
3      4

global_container_movement_disruption_score
global_container_movement_disruption_score
0    171
1     49
2     21
3      1

freight_cost_disruption_score
freight_cost_disruption_score
0    181
1     35
2     19
3      7

supply_disruption_score
supply_disruption_score
0    151
1     51
2     33
3      7

production_disruption_score
production_disruption_score
0    215
1     19
2      5
3      3

covid19_related_disruption_score
covid19_related_disruption_score
0    228
1     13
2      1

demand_disruption_score
demand_disruption_score
0    193
1     28
2     16
3      5

Descriptive statistics


,count,mean,std,min,25%,50%,75%,max
port_disruption_score,242.0,0.491736,0.865387,0.0,0.0,0.000000,1.000000,3.000000
global_container_movement_disruption_score,242.0,0.388430,0.661595,0.0,0.0,0.000000,1.000000,3.000000
freight_cost_disruption_score,242.0,0.388430,0.755307,0.0,0.0,0.000000,0.750000,3.000000
supply_disruption_score,242.0,0.570248,0.833196,0.0,0.0,0.000000,1.000000,3.000000
production_disruption_score,242.0,0.157025,0.499099,0.0,0.0,0.000000,0.000000,3.000000
covid19_related_disruption_score,242.0,0.061983,0.258228,0.0,0.0,0.000000,0.000000,2.000000
demand_disruption_score,242.0,0.309917,0.687035,0.0,0.0,0.000000,0.000000,3.000000
shipping_dimension_max,242.0,0.760331,0.989780,0.0,0.0,0.000000,2.000000,3.000000
shipping_dimension_mean,242.0,0.422865,0.619173,0.0,0.0,0.000000,0.666667,2.333333
shipping_dimension_sum,242.0,1.268595,1.857520,0.0,0.0,0.000000,2.000000,7.000000



Theme score correlation


,port_disruption_score,global_container_movement_disruption_score,freight_cost_disruption_score,supply_disruption_score,production_disruption_score,covid19_related_disruption_score,demand_disruption_score
port_disruption_score,1.000000,0.650646,0.309644,0.559018,0.185549,0.141563,0.112500
global_container_movement_disruption_score,0.650646,1.000000,0.543784,0.672923,0.216637,0.198518,0.473488
freight_cost_disruption_score,0.309644,0.543784,1.000000,0.602620,0.145730,0.131339,0.446727
supply_disruption_score,0.559018,0.672923,0.602620,1.000000,0.552096,0.143606,0.443845
production_disruption_score,0.185549,0.216637,0.145730,0.552096,1.000000,-0.075832,0.111608
covid19_related_disruption_score,0.141563,0.198518,0.131339,0.143606,-0.075832,1.000000,0.195322
demand_disruption_score,0.112500,0.473488,0.446727,0.443845,0.111608,0.195322,1.000000


,article_id,headline,retrieved_publish_date,port_disruption_score,global_container_movement_disruption_score,freight_cost_disruption_score,supply_disruption_score,production_disruption_score,covid19_related_disruption_score,demand_disruption_score,overall_reasoning
24,llm_news_0010590,Port of Long Angeles to develop new container ...,2025-10-03T11:12:41+00:00,0,0,0,0,0,0,0,No taxonomy-defined disruption was identified ...
6,llm_news_0010729,9 strategies every manufacturer needs to know,2025-10-27T05:00:00,0,0,0,1,0,0,1,The article discusses general supply chain vul...
152,llm_news_0009256,Top 10: Supply Chain Accreditations,2024-09-25T11:21:00+00:00,0,0,0,0,0,0,0,No taxonomy-defined disruption was identified ...
209,llm_news_0010477,67 containers fall off Zim ship in Port of Lon...,2025-09-10T03:18:07+00:00,2,1,1,1,0,0,0,The article describes a container ship inciden...
237,llm_news_0011110,Watch: Nissan's Gerardo de la Torre on right-s...,2025-11-24T12:37:01+00:00,1,0,0,2,0,0,0,The article discusses Nissan's right-sizing st...
199,llm_news_0010465,Guangzhou port's half year container throughpu...,2025-09-04T12:10:53+00:00,0,0,0,0,0,0,0,No taxonomy-defined disruption was identified ...
190,llm_news_0009174,Line-towed barges allowed to move to and from ...,2024-09-04T13:00:00+08:00,2,1,0,0,0,0,0,The article describes congestion at Singapore'...
150,llm_news_0009254,Emirates NBD Wins Procurement Transformation a...,2024-09-25T08:00:43+00:00,0,0,0,0,0,1,0,This article focuses on Emirates NBD's procure...
9,llm_news_0009408,Why Estée Lauder Leads the way in Sustainable ...,2024-10-29T15:19:34+00:00,0,0,0,0,0,0,0,No taxonomy-defined disruption was identified ...
113,llm_news_0009225,Why the price of your cup of coffee is set to ...,2024-09-18T04:30:38+00:00,1,1,1,2,3,0,0,This article describes disruptions to coffee p...


## 12. Package the selected chunk outputs

The ZIP contains the final scored CSV, checkpoint, failure table, JSONL audit trail, and run log.

In [12]:
files_to_package = [
    FINAL_OUTPUT_FILE,
    CHECKPOINT_FILE,
    FAILURES_FILE,
    AUDIT_JSONL_FILE,
    RUN_LOG_FILE,
    WORKING_DIR / "chunk_manifest.csv",
]

package_dir = RUN_DIR / "package"
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True, exist_ok=True)

for file_path in files_to_package:
    if file_path.exists():
        shutil.copy2(file_path, package_dir / file_path.name)

zip_path = Path(
    shutil.make_archive(
        str(ARTIFACT_ZIP_BASE),
        "zip",
        root_dir=package_dir,
    )
)

print(f"Packaged artifacts: {zip_path}")
print("Files:")
for file_path in sorted(package_dir.iterdir()):
    print(f"  - {file_path.name}")

Packaged artifacts: /kaggle/working/shipping_disruption_scoring_gemma2_9b/chunk_010_artifacts.zip
Files:
  - chunk_010_audit.jsonl
  - chunk_010_checkpoint.csv
  - chunk_010_failures.csv
  - chunk_010_run.log
  - chunk_manifest.csv
  - scored_chunk_010.csv


## 13. Final consolidation after all 700-row chunks are complete

Place all completed `scored_chunk_*.csv` files in one directory. This section:

- combines the chunk files;
- checks duplicate `row_uid` values;
- verifies row coverage against the original source CSV;
- reports missing or failed rows;
- writes one final scored dataset.

Do not run this section until all run chunks are available.

In [13]:
# # Directory containing all completed scored_chunk_*.csv files.
# COMPLETED_CHUNKS_DIR = Path("/kaggle/input/YOUR-COMPLETED-CHUNKS")
# FINAL_MERGED_OUTPUT = WORKING_DIR / "scored_full_dataset_merged.csv"
# FINAL_MERGE_REPORT = WORKING_DIR / "scored_full_dataset_merge_report.json"

# chunk_files = sorted(COMPLETED_CHUNKS_DIR.glob("scored_chunk_*.csv"))
# if not chunk_files:
#     raise FileNotFoundError(
#         f"No scored_chunk_*.csv files found in {COMPLETED_CHUNKS_DIR}"
#     )

# merged_parts = []
# for file_path in chunk_files:
#     part = pd.read_csv(file_path, low_memory=False)
#     part["_source_chunk_file"] = file_path.name
#     merged_parts.append(part)

# merged = pd.concat(merged_parts, ignore_index=True)

# duplicate_uid_count = int(merged["row_uid"].duplicated().sum())
# if duplicate_uid_count:
#     merged = merged.sort_values(
#         ["source_row_number", "scored_at_utc"],
#         kind="stable",
#     )
#     merged = merged.drop_duplicates(subset="row_uid", keep="last")

# merged = merged.sort_values("source_row_number", kind="stable").reset_index(drop=True)

# expected_row_uids = set(df_all["_row_uid"].astype(str))
# actual_row_uids = set(merged["row_uid"].astype(str))
# missing_row_uids = sorted(expected_row_uids - actual_row_uids)
# unexpected_row_uids = sorted(actual_row_uids - expected_row_uids)

# report = {
#     "created_at_utc": utc_now_iso(),
#     "source_csv_rows": len(df_all),
#     "chunk_files_found": [file_path.name for file_path in chunk_files],
#     "rows_before_deduplication": sum(len(part) for part in merged_parts),
#     "duplicate_row_uid_count": duplicate_uid_count,
#     "merged_rows_after_deduplication": len(merged),
#     "successful_rows": int(merged["scoring_status"].eq("success").sum()),
#     "missing_text_rows": int(merged["scoring_status"].eq("missing_text").sum()),
#     "failed_rows": int(merged["scoring_status"].eq("failed").sum()),
#     "missing_source_rows": len(missing_row_uids),
#     "unexpected_rows": len(unexpected_row_uids),
#     "missing_row_uids": missing_row_uids,
#     "unexpected_row_uids": unexpected_row_uids,
# }

# atomic_write_csv(merged, FINAL_MERGED_OUTPUT)
# FINAL_MERGE_REPORT.write_text(
#     json.dumps(report, ensure_ascii=False, indent=2),
#     encoding="utf-8",
# )

# display(pd.Series(report))
# print(f"Merged dataset: {FINAL_MERGED_OUTPUT}")
# print(f"Merge report: {FINAL_MERGE_REPORT}")

## 14. Methodological notes

1. **700 rows is a run-management chunk, not a model prompt batch.** Full articles vary greatly in length, so each article is scored separately to avoid context overflow and cross-article contamination.
2. **Long articles are not silently truncated.** They are divided into overlapping token windows and the maximum theme severity across the complete article is retained.
3. **Zero means evidence is absent.** Model, parsing, or runtime failures are recorded as `failed` with missing scores, preventing artificial downward bias.
4. **Gemma-specific prompt formatting is used.** The unchanged system rubric is placed at the beginning of the first user turn because Gemma 2 IT uses the user/model turn format. A runtime generation cap prevents the prompt plus output from exceeding Gemma 2's context window.
5. **Primary variables are the seven 0–4 theme scores.** Aggregate maximum, mean, and sum variables are included as transparent derived alternatives; the downstream index-construction rule should be chosen and justified separately.
6. **The source CSV is preserved.** Every original column remains in the scored output, followed by the scoring, evidence, audit, and run-management fields.